# Machine Learning Phase

This notebook extends the statistical engine with an unsupervised machine learning layer for anomaly scoring, representation learning, clustering, and ensemble risk estimation.

## ML Phase Goals

This phase adds a second, unsupervised detection layer on top of the statistical engine.
The workflow answers five practical questions:

1. Which flows look isolated from normal behavior? (Isolation Forest)
2. Can high-dimensional traffic be represented in a compact latent space? (PCA)
3. Do flows form meaningful behavior groups? (K-Means and DBSCAN)
4. How far is each flow from typical behavior? (Euclidean and Mahalanobis distances)
5. How can all signals be combined into one explainable risk score? (ensemble ML risk)

Expected outputs in this notebook:

- Statistical risk baseline (0-100)
- Isolation Forest anomaly score
- PCA behavior map
- Cluster labels and rarity
- Ensemble ML risk and agreement analysis

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src import (
    data_cleaning,
    feature_engineering,
    ml_anomaly,
    ml_clustering,
    ml_representation,
    ml_risk_model,
    risk_index,
    statistical_analysis,
    )

## 1) Data Setup and Synthetic Traffic

This notebook uses a controlled synthetic dataset to keep the ML behavior reproducible.
The data intentionally mixes mostly normal flows with a smaller attack-like segment.

Design choices:

- 92% normal traffic to mimic realistic skew
- 8% anomalous traffic with extreme packet and byte behavior
- Numeric-only feature extraction for stable unsupervised processing

Why this matters:

A controlled dataset helps you verify that anomaly methods react to suspicious patterns before moving to real network logs.

In [ ]:
np.random.seed(42)
n = 2000
normal_data = {
    'Flow Duration': np.random.exponential(500, int(n * 0.92)),
    'Total Fwd Packets': np.random.poisson(8, int(n * 0.92)),
    'Total Backward Packets': np.random.poisson(6, int(n * 0.92)),
    'Total Length of Fwd Packets': np.random.normal(400, 120, int(n * 0.92)),
    'Total Length of Bwd Packets': np.random.normal(300, 100, int(n * 0.92)),
    'Flow Bytes/s': np.random.normal(5000, 2000, int(n * 0.92)),
    'Flow Packets/s': np.random.normal(50, 20, int(n * 0.92)),
    'Fwd Packet Length Mean': np.random.normal(50, 15, int(n * 0.92)),
    'Bwd Packet Length Mean': np.random.normal(45, 12, int(n * 0.92)),
    'Packet Length Mean': np.random.normal(48, 14, int(n * 0.92)),
    'Destination Port': np.random.choice([80, 443, 22, 8080, 3306], int(n * 0.92)),
}
attack_data = {
    'Flow Duration': np.random.exponential(50, int(n * 0.08)),
    'Total Fwd Packets': np.random.poisson(200, int(n * 0.08)),
    'Total Backward Packets': np.random.poisson(2, int(n * 0.08)),
    'Total Length of Fwd Packets': np.random.normal(50000, 10000, int(n * 0.08)),
    'Total Length of Bwd Packets': np.random.normal(100, 50, int(n * 0.08)),
    'Flow Bytes/s': np.random.normal(500000, 100000, int(n * 0.08)),
    'Flow Packets/s': np.random.normal(5000, 1000, int(n * 0.08)),
    'Fwd Packet Length Mean': np.random.normal(250, 50, int(n * 0.08)),
    'Bwd Packet Length Mean': np.random.normal(10, 5, int(n * 0.08)),
    'Packet Length Mean': np.random.normal(230, 45, int(n * 0.08)),
    'Destination Port': np.random.choice([0, 1, 65535, 31337, 4444], int(n * 0.08)),
}

df = pd.concat([pd.DataFrame(normal_data), pd.DataFrame(attack_data)], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
numeric_df = data_cleaning.extract_numeric_data(df)
numeric_df.head()

## 2) Statistical Baseline + Isolation Forest

This step computes two complementary anomaly signals:

- Statistical baseline from Z-score and IQR deviation
- Isolation Forest score from random partitioning in feature space

Interpretation guide:

- Higher `statistical_risk` means stronger deviation from feature-wise norms
- Higher `iforest_score` means stronger model-estimated isolation
- `iforest_flag=True` indicates the model classified the flow as anomalous

Why both are useful:

Statistical scores are transparent and easy to audit, while Isolation Forest captures multivariate structure that simple thresholds can miss.

In [ ]:
scaler = feature_engineering.StandardScaler()
scaled = scaler.fit_transform(numeric_df.values)
zscore_scores = statistical_analysis.compute_deviation_scores(scaled)

iqr_scores = np.zeros(len(numeric_df))
for column in numeric_df.columns:
    lower, upper = statistical_analysis.iqr_bounds(numeric_df[column], multiplier=1.5)
    distance = np.maximum(0, numeric_df[column].values - upper)
    distance = np.maximum(distance, lower - numeric_df[column].values)
    iqr_scores += np.abs(distance)

iqr_scores = feature_engineering.StandardScaler().fit_transform(
    iqr_scores.reshape(-1, 1)
).flatten()
statistical_risk = risk_index.compute_weighted_risk_index(
    {'zscore': zscore_scores, 'iqr': iqr_scores},
    {'zscore': 0.5, 'iqr': 0.5}
 )

iforest_model = ml_anomaly.train_isolation_forest(numeric_df, contamination=0.05, n_estimators=200)
iforest_scores = ml_anomaly.get_anomaly_scores(numeric_df, model=iforest_model)
iforest_flags = ml_anomaly.predict_anomalies(numeric_df, model=iforest_model)

pd.DataFrame({
    'statistical_risk': statistical_risk,
    'iforest_score': iforest_scores * 100,
    'iforest_flag': iforest_flags
}).head()

## 3) Representation Learning with PCA

PCA projects traffic into 2 dimensions while preserving as much variance as possible.
This is primarily for behavior inspection and analyst explainability.

How to read the scatter plot:

- Nearby points have similar traffic patterns
- Isolated points often correspond to unusual flow behavior
- Color intensity reflects Isolation Forest anomaly severity

Operational value:

PCA gives analysts a fast visual triage surface, helping validate whether model alerts align with visible traffic separation.

In [ ]:
pca_result = ml_representation.run_pca(numeric_df, n_components=2)
pca_df = pca_result['components'].copy()
pca_df['statistical_risk'] = statistical_risk
pca_df['iforest_score'] = iforest_scores * 100

fig_pca = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='iforest_score',
    title='PCA traffic map colored by Isolation Forest score',
    color_continuous_scale='Turbo'
 )
fig_pca.show()

## 4) Behavioral Clustering and Distance Deviation

This section creates behavior groups and computes rarity/deviation metrics:

- K-Means: partitions flows into `k` behavior clusters
- DBSCAN: detects dense regions and marks sparse noise (`-1`)
- Euclidean score: distance to nearest cluster center
- Mahalanobis score: covariance-aware distance from global behavior

Why distances matter:

Two points can have similar Euclidean distance but very different covariance context.
Mahalanobis helps identify statistically unusual directions in feature space.

In [ ]:
kmeans_result = ml_clustering.run_kmeans(numeric_df, k=4)
dbscan_result = ml_clustering.run_dbscan(numeric_df)
euclidean_scores = ml_risk_model.euclidean_distance_score(
    kmeans_result['processed_data'],
    kmeans_result['cluster_centers']
 )
mahalanobis_scores = ml_risk_model.mahalanobis_distance_score(
    kmeans_result['processed_data']
 )
distance_scores = (euclidean_scores + mahalanobis_scores) / 2.0
cluster_rarity_scores = np.maximum(
    kmeans_result['rarity_scores'],
    dbscan_result['rarity_scores']
 )
ml_risk = ml_risk_model.compute_ml_risk_score(
    statistical_risk_score=statistical_risk,
    isolation_forest_score=iforest_scores,
    cluster_rarity_score=cluster_rarity_scores,
    distance_deviation_score=distance_scores
 )

cluster_view = pca_df.copy()
cluster_view['cluster'] = kmeans_result['labels'].astype(str)
cluster_view['ml_risk'] = ml_risk
cluster_view['cluster_rarity'] = cluster_rarity_scores

fig_clusters = px.scatter(
    cluster_view,
    x='PC1',
    y='PC2',
    color='cluster',
    size='ml_risk',
    title='Behavioral clustering in PCA space'
 )
fig_clusters.show()

## 5) Ensemble Risk and Agreement Analysis

The final table combines four signals into one normalized ML risk score (0-100):

- Statistical baseline risk
- Isolation Forest anomaly score
- Cluster rarity score
- Distance deviation score

Then we compare statistical and ML decisions using a threshold of 50.

Interpretation:

- `Agreement`: both layers classify similarly
- `Disagreement`: one layer detects risk the other does not

Disagreements are often the most informative rows to investigate first.

In [ ]:
comparison_df = pd.DataFrame({
    'statistical_risk': statistical_risk,
    'iforest_score': iforest_scores * 100,
    'cluster_rarity': cluster_rarity_scores * 100,
    'distance_score': distance_scores * 100,
    'ml_risk': ml_risk,
    'kmeans_cluster': kmeans_result['labels'],
    'dbscan_cluster': dbscan_result['labels']
})
comparison_df['statistical_flag'] = comparison_df['statistical_risk'] >= 50
comparison_df['ml_flag'] = comparison_df['ml_risk'] >= 50
comparison_df['agreement'] = np.where(
    comparison_df['statistical_flag'] == comparison_df['ml_flag'],
    'Agreement',
    'Disagreement'
 )
comparison_df.sort_values('ml_risk', ascending=False).head(20)

## Notes

- Isolation Forest supplies unsupervised anomaly pressure without labels.
- PCA exposes traffic behavior in a 2D latent space for visual inspection.
- K-Means and DBSCAN reveal dense traffic regimes and sparse rare behavior.
- The final ensemble ML risk score blends statistical, anomaly, rarity, and distance signals into a normalized 0-100 score.